# Codigo que cuenta los correos recibidos por organización

In [5]:

import sqlite3

# Conectarse a la base de datos emaildb2.sqlite.
# Si el archivo no existe, SQLite lo crea automáticamente.
conn = sqlite3.connect('emaildb2.sqlite')

# Crear un cursor para ejecutar comandos SQL.
cur = conn.cursor()

# Eliminar la tabla Counts si ya existe.
# Esto permite comenzar el conteo desde cero.
cur.execute('''
DROP TABLE IF EXISTS Counts''')

# Crear la tabla Counts con dos columnas:
# org: almacena el dominio de la organización.
# count: almacena el número de mensajes de ese dominio.
cur.execute('''
CREATE TABLE Counts (org TEXT, count INTEGER)''')

# Solicitar al usuario el nombre del archivo que contiene los correos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utilizar mbox.txt.
if (len(fname) < 1):
    fname = 'mbox.txt'

# Abrir el archivo para leerlo.
fh = open(fname)

# Recorrer el archivo línea por línea.
for line in fh:

    # Ignorar las líneas que no comienzan con 'From: '.
    # continue hace que Python pase a la siguiente línea.
    if not line.startswith('From: '):
        continue

    # Dividir la línea en palabras usando los espacios.
    pieces = line.split()

    # Obtener el correo electrónico, que está en la segunda posición.
    email = pieces[1]

    # Encontrar la posición del símbolo @ en el correo electrónico.
    # Por ejemplo, en 'ana@gmail.com', find('@') devuelve 3.
    dom = email.find('@')

    # Extraer todo lo que aparece después del símbolo @.
    # Por ejemplo, 'ana@gmail.com' se convierte en 'gmail.com'.
    org = email[dom + 1:len(email)]

    # Buscar en la tabla Counts si el dominio ya está registrado.
    # El signo ? es un marcador para el valor que se pasa en (org,).
    cur.execute('SELECT count FROM Counts WHERE org = ? ', (org,))

    # Recuperar la primera fila del resultado.
    # Si el dominio no existe en la tabla, row será None.
    row = cur.fetchone()

    # Si no existe ningún registro para ese dominio...
    if row is None:

        # Insertar el dominio en la tabla con un contador inicial de 1.
        cur.execute('''INSERT INTO Counts (org, count)
                VALUES (?, 1)''', (org,))

    else:

        # Si el dominio ya existe, incrementar su contador en 1.
        # WHERE org = ? garantiza que solo se actualice ese dominio.
        cur.execute('UPDATE Counts SET count = count + 1 WHERE org = ?',
                    (org,))

# Guardar los cambios realizados en la base de datos.
conn.commit()

# Consulta SQL para seleccionar las 10 organizaciones más frecuentes.
# ORDER BY count DESC ordena de mayor a menor.
# LIMIT 10 devuelve como máximo 10 filas.
# Referencia: https://www.sqlite.org/lang_select.html
sqlstr = 'SELECT org, count FROM Counts ORDER BY count DESC LIMIT 10'

# Ejecutar la consulta y recorrer los resultados.
for row in cur.execute(sqlstr):

    # Imprimir el dominio de la organización y su número de apariciones.
    # row[0] contiene el dominio y row[1] contiene el contador.
    print(str(row[0]), row[1])

# Cerrar el archivo, el cursor y la conexión con la base de datos.
fh.close()
cur.close()
conn.close()

Enter file name: 
iupui.edu 536
umich.edu 491
indiana.edu 178
caret.cam.ac.uk 157
vt.edu 110
uct.ac.za 96
media.berkeley.edu 56
ufp.pt 28
gmail.com 25
et.gatech.edu 17
